In [ ]:
import numpy as np
arrNew = [
[0, 0, 0, 0, 0, 0, 0, 17, 0, 0, 0, 0, 14],
[0, 0, 0, 13, 13, 13, 0, 0, 0, 0, 0, 12, 0],
[0, 0, 0, 0, 0, 0, 0, 17, 3, 17, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 16, 0, 0, 0, 0, 14, 0],
[6, 6, 17, 0, 0, 16, 0, 0, 0, 0, 12, 0, 0],
[0, 0, 0, 5, 16, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 10, 14, 0, 0, 0],
[0, 0, 9, 0, 0, 0, 0, 2, 0, 0, 11, 11, 8],
[0, 15, 0, 0, 0, 0, 10, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 16, 16, 10, 0, 0, 0, 0, 0, 0, 0],
[0, 15, 0, 0, 0, 0, 0, 11, 15, 1, 0, 0, 0],
[9, 0, 0, 0, 0, 15, 0, 0, 0, 0, 0, 0, 0]
]

arrNew=np.array(arrNew)
locArr=[] # locArr stores the indices of where a specific integer K is present
# index 0 holds the locations of the integer 1, index 2 holds the locations of the integer 2, and so on
for i in range(1,18):
  locArr.append(np.column_stack(np.where(arrNew==i)).tolist())

print(len(locArr[3]))
print(len(locArr[6]))

In [ ]:
def checkBounds(topI,topJ,botI,botJ):
  if (topI<0 or topJ<0):
    return False
  if (botI>12 or botJ>12):
    return False
  return True


In [ ]:
def coordToShape(arr): # Turns a list of coordinates of numbers into a numpy array representing the k-omino
# For example, [(0,0),(1,0),(1,1)] becomes [[1,0],[1,1]]
  minI = min(a[0] for a in arr)
  minJ = min(b[1] for b in arr)
  for i in range(len(arr)):
    arr[i][0]-=minI
    arr[i][1]-=minJ
  maxI = max(c[0] for c in arr)
  maxJ = max(d[1] for d in arr)
  shapeArr=np.zeros((maxI+1,maxJ+1))
  for coord in arr:
    shapeArr[tuple(coord)]=1
  return shapeArr

In [ ]:
import copy
import matplotlib.pyplot as plt
from collections import Counter
import numpy as np

def overlayShape(gridArr,shapeArr, numAnchor,numLocs):
  # numAnchor is the number we want to overlay our shape onto
  # numLocs is the list of all locations of a given integer on the original grid
  # overlayShape returns True if shapeArr can fit on gridArr, and False otherwise
  # otherwise
  results=[]
  shapeCoords=[]
  coordArr=[]
  for i in range(len(shapeArr)):
    for j in range(len(shapeArr[0])):
      if (shapeArr[i,j]):
        coordArr.append([i,j])

  for coordinate in coordArr:
    ival = coordinate[0]
    jval = coordinate[1]
    IVal = numLocs[0][0]
    JVal = numLocs[0][1]
    topI = IVal-ival
    topJ = JVal-jval
    botI = topI+len(shapeArr)-1
    botJ = topJ + len(shapeArr[0])-1

    if not checkBounds(topI, topJ, botI, botJ):
        continue

    newCoords = [[c[0] + topI, c[1] + topJ] for c in coordArr]

    if not all(loc in newCoords for loc in numLocs):
        continue

    valid = True
    for i, j in newCoords:
        if gridArr[i, j] not in (0, numAnchor):
            valid = False
            break

    if valid:
        return True

  return False



In [ ]:
import numpy as np
import copy
import time

# The final version. Duplicates are filtered out, we check to see if the shape can fit,
# and we track all k-ominoes that can make the k+1 omino.

def bestGenerateKOmino(wholeStore,num):
    global arrNew
    global locArr
    finSet=dict()
    retArr=[]
    for s in range(len(wholeStore)):
      shape=wholeStore[s][0]
      direcI = [0,1,0,-1]
      direcJ = [1,0,-1,0]

      coordArr=[]

      for i in range(len(shape)):
        for j in range(len(shape[0])):
          if (shape[i,j]):
            coordArr.append([i,j])

      for block in coordArr:
        for direc in range(4):
            newCoords = copy.deepcopy(coordArr)
            newI=block[0]+direcI[direc]
            newJ=block[1]+direcJ[direc]
            if ([newI,newJ] not in newCoords):

              newCoords.append([newI,newJ])

              newShape = coordToShape(newCoords)

              symmetries = []

              rot = newShape

              for _ in range(4):
                rot = np.rot90(rot)
                symmetries.append(rot)
                symmetries.append(np.flip(rot, 0))

              for finalSym in symmetries:
                key = (finalSym.tobytes(),finalSym.shape)
                if (num in (4,7)):
                  if (overlayModified(arrNew,finalSym,num)):
                    if (key not in finSet):
                      ref = (finalSym,{s})
                      finSet[key]=ref
                      retArr.append(ref)
                    else:
                      finSet[key][1].add(s)

                elif (overlayShape(arrNew,finalSym,num,locArr[num-1])):
                  if (key not in finSet):
                    ref = (finalSym,{s})
                    finSet[key]=ref
                    retArr.append(ref)
                  else:
                    finSet[key][1].add(s)

    return retArr

res=[np.array([[[1]]])]
allShapes=[]
allShapes.append(res)
for i in range(16):
  res = bestGenerateKOmino(res,i+2)
  allShapes.append(res)
  print(i+2,len(res))

  # all free spaces move grid around shape

In [ ]:
import copy
import matplotlib.pyplot as plt
from collections import Counter
import numpy as np

def overlayModified(gridArr,shapeArr, numAnchor):
  freeSpots = np.column_stack(np.where(gridArr==0)).tolist()
  results=[]
  shapeCoords=[]
  coordArr=[]
  for i in range(len(shapeArr)):
    for j in range(len(shapeArr[0])):
      if (shapeArr[i,j]):
        coordArr.append([i,j])

  ival = coordArr[0][0]
  jval = coordArr[0][1]

  for freeSpot in freeSpots:
    IVal = freeSpot[0]
    JVal = freeSpot[1]
    topI = IVal-ival
    topJ = JVal-jval
    botI = topI+len(shapeArr)-1
    botJ = topJ + len(shapeArr[0])-1

    if not checkBounds(topI, topJ, botI, botJ):
        continue

    newCoords = [[c[0] + topI, c[1] + topJ] for c in coordArr]

    valid = True
    for i, j in newCoords:
        if gridArr[i, j] !=0:
            valid = False
            break

    if valid:
        return True

  return False



In [ ]:
import copy
import matplotlib.pyplot as plt
import numpy as np

# This method adds a shape to a given grid and returns a copy of all possible modified grids

def overlayShapesModified(gridArr,shapeArr, numAnchor):
  results=[]
  coordArr=[]
  for i in range(len(shapeArr)):
    for j in range(len(shapeArr[0])):
      if (shapeArr[i,j]):
        coordArr.append([i,j])
  gridCopy=copy.deepcopy(gridArr)

  freeSpots = np.column_stack(np.where(gridArr==0)).tolist()
  ival = coordArr[0][0]
  jval = coordArr[0][1]

  for freeSpot in freeSpots:
    useGrid = copy.deepcopy(gridCopy)

    IVal = freeSpot[0]
    JVal = freeSpot[1]

    topI = IVal-ival
    topJ = JVal-jval
    botI = topI+len(shapeArr)-1
    botJ = topJ + len(shapeArr[0])-1

    if (checkBounds(topI,topJ,botI,botJ)):
      newCoords=[]
      for coord in coordArr:
        newCoords.append([coord[0]+topI,coord[1]+topJ])
      flag=False


      for cord in newCoords:
        if (useGrid[cord[0],cord[1]] in (0,numAnchor)):
          flag=True
          useGrid[cord[0],cord[1]]=numAnchor
        else:
          flag=False
          break

      if (flag):
        results.append(useGrid)

  return results





In [ ]:
import copy
import matplotlib.pyplot as plt
import numpy as np

# This method adds a shape to a given grid and returns a copy of all possible modified grids

def overlayShapes(gridArr,shapeArr, numAnchor,numLoca):
  results=[]
  coordArr=[]
  for i in range(len(shapeArr)):
    for j in range(len(shapeArr[0])):
      if (shapeArr[i,j]):
        coordArr.append([i,j])
  gridCopy=copy.deepcopy(gridArr)

  for coordinate in coordArr:
    useGrid = copy.deepcopy(gridCopy)
    ival = coordinate[0]
    jval = coordinate[1]
    IVal = numLoca[0][0]
    JVal = numLoca[0][1]

    topI = IVal-ival
    topJ = JVal-jval
    botI = topI+len(shapeArr)-1
    botJ = topJ + len(shapeArr[0])-1

    if (checkBounds(topI,topJ,botI,botJ)):
      newCoords=[]
      for coord in coordArr:
        newCoords.append([coord[0]+topI,coord[1]+topJ])
      flag=False

      if not all(loc in newCoords for loc in numLoca):
        continue

      for cord in newCoords:
        if (useGrid[cord[0],cord[1]] in (0,numAnchor)):
          flag=True
          useGrid[cord[0],cord[1]]=numAnchor
        else:
          flag=False
          break

      if (flag):
        results.append(useGrid)

  return results





In [ ]:
def labelArr(arr):
  for i in range(arr.shape[0]):
      for j in range(arr.shape[1]):
          plt.text(j, i, arr[i, j], ha="center", va="center", color="black")


In [ ]:
plt.imshow(arrNew, interpolation="nearest",cmap = 'coolwarm')
labelArr(arrNew) # Places numbers on the grid
plt.show()

savG=None
def fillGridKOmino(grid,indSet,num):
  global allShapes
  global savG
  if (num==2): # This is when all shapes have been added to the grid
    savG=grid
    return

  if (num in (5,8)): # num is the lowest positive integer on the board.
  # If we want to add a 4 or 7-omino, we have to check if num is 5 or 8 respectively

    for indices in indSet:
      indShape = allShapes[num-2][indices][0]

      if (overlayModified(grid,indShape,num-1)):

        resGrid = overlayShapesModified(grid,indShape,num-1)

        indexSet = allShapes[num-2][indices][1]
        for resG in resGrid:
          fillGridKOmino(resG,indexSet,num-1)

  else:
    for indices in indSet:
      indShape = allShapes[num-2][indices][0]

      if (overlayShape(grid,indShape,num-1,locArr[num-2])):

        resGrid = overlayShapes(grid,indShape,num-1,locArr[num-2])

        indexSet = allShapes[num-2][indices][1]
        for resG in resGrid:
          fillGridKOmino(resG,indexSet,num-1)


for shape in allShapes[-1]:
  tryAddShape=overlayShapes(arrNew,shape[0],17,locArr[-1])
  indPass = shape[1]
  for grids in tryAddShape:
    fillGridKOmino(grids,indPass,17)

plt.imshow(savG, interpolation="nearest",cmap = 'coolwarm')
labelArr(savG)
plt.show()
